# Yuki F5-TTS on Colab

Este notebook sobe um servidor HTTP com F5-TTS no Colab e expõe uma URL publica para a Yuki usar.

In [ ]:
!nvidia-smi
!pip install -q f5-tts fastapi uvicorn python-multipart nest-asyncio requests

In [ ]:
from google.colab import files
import os

os.makedirs('/content/ref_audio', exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Envie o audio de referencia antes de continuar.')

uploaded_name = next(iter(uploaded))
REF_AUDIO = f'/content/ref_audio/{uploaded_name}'
with open(REF_AUDIO, 'wb') as f:
    f.write(uploaded[uploaded_name])

REF_TEXT = ''
print('REF_AUDIO =', REF_AUDIO)
print('REF_TEXT vazio = transcricao automatica pelo F5-TTS.')

In [ ]:
%%writefile /content/f5_server.py
import os
import tempfile
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from f5_tts.api import F5TTS

REF_AUDIO = os.environ['REF_AUDIO']
REF_TEXT = os.environ['REF_TEXT']
MODEL = os.environ.get('F5_MODEL', 'F5TTS_v1_Base')
tts = F5TTS(model=MODEL)
app = FastAPI()

class TTSRequest(BaseModel):
    text: str
    speed: float = 1.0
    remove_silence: bool = False

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL}

@app.post('/tts')
def tts_endpoint(req: TTSRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail='text is empty')

    fd, out_path = tempfile.mkstemp(suffix='.wav')
    os.close(fd)
    tts.infer(
        ref_file=REF_AUDIO,
        ref_text=REF_TEXT,
        gen_text=req.text,
        speed=req.speed,
        remove_silence=req.remove_silence,
        file_wave=out_path,
        seed=None,
    )
    return FileResponse(out_path, media_type='audio/wav', filename='yuki_f5.wav')

In [ ]:
import os
import subprocess
import time
import nest_asyncio

nest_asyncio.apply()
os.environ['REF_AUDIO'] = REF_AUDIO
os.environ['REF_TEXT'] = REF_TEXT

server = subprocess.Popen(['uvicorn', 'f5_server:app', '--host', '0.0.0.0', '--port', '9880'], cwd='/content')
for _ in range(30):
    time.sleep(2)
    try:
        import requests
        r = requests.get('http://127.0.0.1:9880/health', timeout=5)
        if r.ok:
            print('Servidor F5-TTS ativo:', r.json())
            break
    except Exception:
        pass
else:
    raise RuntimeError('Servidor F5-TTS nao iniciou.')

In [ ]:
import re
import subprocess
import time
from pathlib import Path

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

log_path = Path('/content/cloudflared.log')
with open(log_path, 'w') as log:
    tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:9880'], stdout=log, stderr=log)

public_url = None
for _ in range(60):
    time.sleep(2)
    text = log_path.read_text(encoding='utf-8', errors='ignore')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError('Nao consegui obter a URL publica do tunnel.')

import requests
for _ in range(15):
    try:
        health = requests.get(public_url + '/health', timeout=10)
        if health.ok:
            print('Health remoto OK:', health.json())
            break
    except Exception:
        time.sleep(2)

print('COLE NO conf.yaml:')
print(f"  tts_model: 'f5_tts'")
print(f"  api_url: '{public_url}/tts'")
print(f"  health: '{public_url}/health'")

In [ ]:
import time
while True:
    print('Yuki F5-TTS online:', time.strftime('%H:%M:%S'))
    time.sleep(60)